# 01 — Frozen splits

Rebuilds the three split manifests on Kaggle and checks they are **identical** to the ones committed in `manifests/`.
If they match, the splits are reproducible from the raw data and we never touch them again.

Setup: dataset `paultimothymooney/chest-xray-pneumonia`, Internet **on**, accelerator **None**.

In [ ]:
!pip install -q --no-deps --force-reinstall --no-cache-dir git+https://github.com/meap11/ppneu.git

In [ ]:
import json, os
from pathlib import Path
import pandas as pd
from ppneu import audit as A, splits as S

root, _ = A.find_root("/kaggle/input")
index = A.build_index(root, workers=os.cpu_count())   # same as notebook 00 (~3 min)
manifests = S.build_all(index, seed=0)
for name, m in manifests.items():
    print(name, json.dumps(S.leakage_report(m)))

In [ ]:
RAW = "https://raw.githubusercontent.com/meap11/ppneu/main/manifests/{}.csv"
for name, m in manifests.items():
    ref = pd.read_csv(RAW.format(name))
    same = ref.reset_index(drop=True).equals(m.reset_index(drop=True).astype(ref.dtypes.to_dict()))
    print(f"{name:13s} identical to committed manifest: {same}")
    assert same, f"{name} differs from the committed manifest"

In [ ]:
# Every manifest path must exist on disk
for name, m in manifests.items():
    missing = [p for p in m["relpath"] if not (root / p).exists()]
    print(name, "missing files:", len(missing))